# 第3章　決定木の理論とアルゴリズム

## 3.4　決定木のアルゴリズム

##3.4.1　CARTによる分類木アルゴリズム

### コード3.1　PythonでCARTによる分類木構築（Step 0）


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

### コード3.2　PythonでCARTによる分類木構築（Step 1）


In [2]:
def load_data():
    # irisデータセットを読み込む
    df = sns.load_dataset('iris')
    # 説明変数（がく片，花弁の長さ・幅）を取り出す
    X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
    # ターゲット変数（種類）を0,1,2の数値に変換する
    y = df['species'].astype('category').cat.codes.values
    # データを訓練用とテスト用に8:2で分割（stratify=yでクラス比率は維持する）
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y
    )
    return X_train, X_test, y_train, y_test

### コード3.3　PythonでCARTによる分類木構築（Step 2）

In [3]:
def gini_impurity(y):
    # ジニ不純度を計算する（単一クラスのみなら 0）
    m = len(y)
    if m == 0:
        return 0
    return 1.0 - sum((np.sum(y == c) / m) ** 2 for c in np.unique(y))

def init_node(y):
    # ユニークなクラスの種類（例: [0,1,2]）を取得
    classes = np.unique(y)
    # 各クラスのサンプル数を数える
    num_samples_per_class = [np.sum(y == i) for i in classes]
    # サンプル数が最多のクラスを予測クラスとする
    predicted_class = classes[np.argmax(num_samples_per_class)]
    # ノードの情報を辞書にまとめる
    node = {
        'gini': gini_impurity(y),           # ジニ不純度（データの混ざり具合）
        'num_samples': len(y),              # ノードに含まれるデータ数
        'num_samples_per_class': num_samples_per_class,  # クラスごとの数
        'predicted_class': predicted_class, # 最多クラス（リーフノードなら予測ラベル）
        'is_leaf': False,                   # リーフノードかどうか
        'feature_index': None,              # 分割する特徴のインデックス
        'threshold': None,                  # 分割のしきい値
        'left': None,                       # 左の子ノード
        'right': None                       # 右の子ノード
    }
    return node

### コード3.4　PythonでCARTによる分類木構築（Step 3）

In [4]:
def best_split(X, y):
    m, n = X.shape
    if m <= 1:
        return None, None
    best_gini = 1.0  # 最良のジニ不純度
    best_idx, best_thr = None, None

    classes_unique = np.unique(y)
    class_to_index = {c: i for i, c in enumerate(classes_unique)}  # クラス番号を0始まりに変換

    # 各特徴量で最良の分割点を探す
    for idx in range(n):
        thresholds, classes = zip(*sorted(zip(X[:, idx], y)))
        num_left = [0] * len(classes_unique)
        num_right = [np.sum(y == c) for c in classes_unique]
        for i in range(1, m):
            c = classes[i - 1]
            c_idx = class_to_index[c]
            num_left[c_idx] += 1
            num_right[c_idx] -= 1
            gini_left = 1.0 - sum((num_left[x] / i) ** 2 for x in range(len(classes_unique)))
            gini_right = 1.0 - sum((num_right[x] / (m - i)) ** 2 for x in range(len(classes_unique)))
            gini = (i * gini_left + (m - i) * gini_right) / m
            if thresholds[i] == thresholds[i - 1]:
                continue
            if gini < best_gini:
                best_gini = gini
                best_idx = idx
                best_thr = (thresholds[i] + thresholds[i - 1]) / 2
    return best_idx, best_thr

### コード3.5　PythonでCARTによる分類木構築（Step 4）

In [5]:
def build_tree(X, y):
    # ノードの情報を作成する
    node = init_node(y)

    # 1. ノードが純粋(gini=0)なら，リーフノードとして終了
    if node['gini'] == 0:
        node['is_leaf'] = True
        return node

    # 分割できるか確認する
    idx, thr = best_split(X, y)

    if idx is not None:
        # 左右にデータを分ける
        indices_left = X[:, idx] < thr
        X_left, y_left = X[indices_left], y[indices_left]
        X_right, y_right = X[~indices_left], y[~indices_left]

        # 分割した結果，片方が空になった場合もリーフとする
        if len(y_left) == 0 or len(y_right) == 0:
            node['is_leaf'] = True
            return node

        node['feature_index'] = idx
        node['threshold'] = thr
        # 左右の子ノードを再帰的に作成する
        node['left'] = build_tree(X_left, y_left)
        node['right'] = build_tree(X_right, y_right)
    else:
        # 分割できない場合はリーフノードとする (m <= 1 または全特徴量が同一)
        node['is_leaf'] = True

    return node

### コード3.6　PythonでCARTによる分類木構築（Step 5）

In [6]:
def count_leaves(node):
    # ノード次のリーフノード数を数える
    if node['is_leaf']:
        return 1
    return count_leaves(node['left']) + count_leaves(node['right'])

def total_impurity(node):
    # ノード次のリーフノードのジニ不純度の合計を計算する
    if node['is_leaf']:
        return node['gini'] * node['num_samples']
    return total_impurity(node['left']) + total_impurity(node['right'])

def compute_alpha(node):
    # ノードのプルーニング用指標αを計算する
    if node['is_leaf']:
        return np.inf
    R_t = node['gini'] * node['num_samples']
    R_T_t = total_impurity(node)
    T_t_leaves = count_leaves(node)
    if T_t_leaves <= 1:
        return np.inf
    alpha_t = (R_t - R_T_t) / (T_t_leaves - 1)
    return alpha_t

def prune_tree(node, alpha_threshold):
    # αが閾値以下ならノードをプルーニング（リーフ化）する
    # 後順 (Post-order) 探索
    # 1. ベースケース：既にリーフなら何もしない
    if node['is_leaf']:
        return
    # 2. 先に子ノードを再帰的に処理する
    prune_tree(node['left'], alpha_threshold)
    prune_tree(node['right'], alpha_threshold)
    # 3. 子ノードの処理後に現在のノードのαを計算する
    alpha = compute_alpha(node)
    # 4. αが閾値以下なら，このノードを剪定する
    if alpha <= alpha_threshold:
        node['left'] = None
        node['right'] = None
        node['is_leaf'] = True

def find_min_alpha(node):
    # ツリー全体の最小のαを見つける
    if node['is_leaf']:
        return np.inf
    # 現在のノードのαを計算
    current_alpha = compute_alpha(node)
    # 左右の子ノードの最小αを再帰的に見つける
    left_min_alpha = find_min_alpha(node['left'])
    right_min_alpha = find_min_alpha(node['right'])
    # 現在のノードのαと左右の子ノードの最小αを比較し，最小値を返す
    return min(current_alpha, left_min_alpha, right_min_alpha)

### コード3.7　PythonでCARTによる分類木構築（Step 6）

In [7]:
def predict_input(x, node):
    # 1件のデータを分類木に通し，予測クラスを返す
    while not node['is_leaf']:
        if x[node['feature_index']] < node['threshold']:
            node = node['left']
        else:
            node = node['right']
    return node['predicted_class']

def accuracy(X, y, tree):
    # データセット全体の精度を計算する
    predictions = [predict_input(x, tree) for x in X]
    return np.mean(predictions == y)

### コード3.8　PythonでCARTによる分類木構築（実行コード）

In [8]:
# データ準備
X_train, X_test, y_train, y_test = load_data()
# 木の構築
tree = build_tree(X_train, y_train)
# プルーニング前の精度評価
acc_train_before = accuracy(X_train, y_train, tree)
acc_test_before = accuracy(X_test, y_test, tree)
print(f'Before pruning - Train accuracy: {acc_train_before:.2f}, Test accuracy: {acc_test_before:.2f}')
# プルーニング（最小のαを使う）
min_alpha = find_min_alpha(tree)
prune_tree(tree, min_alpha)
# プルーニング後の精度評価
acc_train_after = accuracy(X_train, y_train, tree)
acc_test_after = accuracy(X_test, y_test, tree)
print(f'After pruning  - Train accuracy: {acc_train_after:.2f}, Test accuracy: {acc_test_after:.2f}')

Before pruning - Train accuracy: 1.00, Test accuracy: 0.90
After pruning  - Train accuracy: 0.99, Test accuracy: 0.93


### コード3.9　scikit-learn による分類木の実装

In [9]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Step 1: データの準備 (変更なし)
def load_data():
    # irisデータセットを読み込む
    df = sns.load_dataset('iris')
    # 説明変数（がく片，花弁の長さ・幅）を取り出す
    X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].values
    # ターゲット変数（種類）を0,1,2の数値に変換する
    y = df['species'].astype('category').cat.codes.values
    # データを訓練用とテスト用に8:2で分割（クラス比率は維持する）
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y
    )
    return X_train, X_test, y_train, y_test

if __name__ == "__main__":
    # データ準備
    X_train, X_test, y_train, y_test = load_data()

    # ----------------------------------------------------------------------
    # Step A: 最大の木（プルーニング前）の構築と評価
    # ----------------------------------------------------------------------

    # 1. CART（Gini不純度）による最大の木の構築 (ccp_alpha=0.0で剪定なし)
    tree_before = DecisionTreeClassifier(
        criterion='gini',
        ccp_alpha=0.0,
        random_state=42
    )
    tree_before.fit(X_train, y_train)

    # プルーニング前の精度評価
    acc_train_before = accuracy_score(y_train, tree_before.predict(X_train))
    acc_test_before = accuracy_score(y_test, tree_before.predict(X_test))

    # 出力形式に合わせたprint文
    print(f'Before pruning - Train accuracy: {acc_train_before:.2f}, Test accuracy: {acc_test_before:.2f}')

    # ----------------------------------------------------------------------
    # Step B: 最小のα（min_alpha）を用いた剪定と評価
    # ----------------------------------------------------------------------

    # 2. 剪定パスを計算し，最初の非ゼロのα（min_alpha）を取得
    path = tree_before.cost_complexity_pruning_path(X_train, y_train)
    ccp_alphas = path.ccp_alphas

    # ccp_alphas[1]が，手動実装の find_min_alpha の結果に相当する最小のα
    min_alpha_sklearn = ccp_alphas[1] if len(ccp_alphas) > 1 else 0.0

    # 3. 最小のαを用いて木を再構築（剪定の適用）
    tree_after = DecisionTreeClassifier(
        criterion='gini',
        ccp_alpha=min_alpha_sklearn, # 最小のαで剪定（CCPの最初のステップ）
        random_state=42
    )
    tree_after.fit(X_train, y_train)

    # 剪定後の木の精度評価
    acc_train_after = accuracy_score(y_train, tree_after.predict(X_train))
    acc_test_after = accuracy_score(y_test, tree_after.predict(X_test))

    # 出力形式に合わせたprint文
    print(f'After pruning - Train accuracy: {acc_train_after:.2f}, Test accuracy: {acc_test_after:.2f}')

Before pruning - Train accuracy: 1.00, Test accuracy: 0.93
After pruning - Train accuracy: 0.99, Test accuracy: 0.97


## 3.4.2　CARTによる回帰木アルゴリズム

### コード3.10　PythonでCARTによる回帰木構築（Step 0）

In [10]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

### コード3.11　PythonでCARTによる回帰木構築（Step 1）

In [11]:
def load_data():
    df = sns.load_dataset('diamonds')
    df_encoded = pd.get_dummies(df, columns=['cut', 'color', 'clarity'], drop_first=True)
    feature_cols = [col for col in df_encoded.columns if col != 'price']
    X = df_encoded[feature_cols].values
    y = np.log(df_encoded['price'].values)  # priceをlog変換する
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, shuffle=True
    )
    return X_train, X_test, y_train, y_test

### コード3.12　PythonでCARTによる回帰木構築（Step 2）

In [12]:
def mse(y):
    # サンプル集合 y の平均二乗誤差を計算する
    if len(y) == 0:
        return 0  # データが空の場合は0を返す
    return np.mean((y - np.mean(y)) ** 2)

def init_node(y):
    # ノードを初期化し，必要な情報を格納する
    predicted_value = np.mean(y)  # ノードに属するデータの平均値を暫定予測値とする
    node = {
        'mse': mse(y),                # ノード内の平均二乗誤差
        'num_samples': len(y),        # ノード内のサンプル数
        'predicted_value': predicted_value,  # ノードの予測値（平均値）
        'is_leaf': False,             # リーフノードフラグ
        'feature_index': None,        # 分割に使用した特徴量インデックス
        'threshold': None,            # 分割閾値
        'left': None,                 # 左側の子ノード
        'right': None                 # 右側の子ノード
    }
    return node

### コード3.13　PythonでCARTによる回帰木構築（Step 3）

In [13]:
def best_split(X, y):
    # データ数と特徴量数を取得
    m, n = X.shape
    if m <= 1:
        # データが1つ以下の場合は分割を行わない
        return None, None
    best_mse = np.inf  # 最小MSEの初期値を∞に設定
    best_idx, best_thr = None, None  # 最適な特徴量インデックスと閾値の初期化

    # 全特徴量を順に評価
    for idx in range(n):
        # 対象の特徴量でデータをソート
        sorted_indices = np.argsort(X[:, idx])
        thresholds = X[sorted_indices, idx]
        values = y[sorted_indices]
        # 境界値候補を順に試行
        for i in range(1, m):
            # 同じ値の境界はスキップ
            if thresholds[i] == thresholds[i - 1]:
                continue
            # 左右の子ノードのターゲット変数を分割
            y_left, y_right = values[:i], values[i:]
            # 左右の子ノードの MSE を計算
            mse_left = mse(y_left)
            mse_right = mse(y_right)
            # 子ノードのデータ数を重みとした加重平均 MSE を計算
            weighted_mse = (i * mse_left + (m - i) * mse_right) / m
            # 最小の MSE となる分割を記録
            if weighted_mse < best_mse:
                best_mse = weighted_mse
                best_idx = idx
                best_thr = (thresholds[i] + thresholds[i - 1]) / 2  # 閾値は中間値を使用
    return best_idx, best_thr  # 最適な特徴量インデックスと閾値を返す

### コード3.14　PythonでCARTによる回帰木構築（Step 4）

In [14]:
def build_tree(X, y, depth=0, max_depth=15, min_samples_split=5):  # max_depth を15に設定
    # 現在のノードを初期化（平均値，MSE，サンプル数などを格納）
    node = init_node(y)

    # 終了条件の確認：最大深さに達したか，サンプル数が最小分割数未満か
    if depth >= max_depth or len(y) < min_samples_split:
        node['is_leaf'] = True  # リーフノードとマーク
        return node

    # 最適な分割を探索
    idx, thr = best_split(X, y)
    if idx is not None:
        # データを左右の子ノードに分割
        indices_left = X[:, idx] < thr
        X_left, y_left = X[indices_left], y[indices_left]
        X_right, y_right = X[~indices_left], y[~indices_left]

        # 片方が空ならこれ以上分割できないのでリーフノードとする
        if len(y_left) == 0 or len(y_right) == 0:
            node['is_leaf'] = True
            return node

        # 分割情報を記録
        node['feature_index'] = idx
        node['threshold'] = thr

        # 左右の子ノードを再帰的に作成（深さを1増やす）
        node['left'] = build_tree(X_left, y_left, depth + 1, max_depth, min_samples_split)
        node['right'] = build_tree(X_right, y_right, depth + 1, max_depth, min_samples_split)
    else:
        # 分割できない場合はリーフノードとする
        node['is_leaf'] = True

    return node  # 構築したノードを返す

### コード3.15　PythonでCARTによる回帰木構築（Step 5）

In [15]:
def count_leaves(node):
    # 与えられたノード以下に含まれるリーフノード数を計算する
    if node['is_leaf']:
        return 1  # リーフノードの場合は1を返す
    # 左右部分木に対して再帰的に呼び出し，リーフノード数を合計する
    return count_leaves(node['left']) + count_leaves(node['right'])

def total_mse(node):
    # 与えられたノード以下の部分木における総二乗誤差（MSEの総和）を計算する
    if node['is_leaf']:
        return node['mse'] * node['num_samples']  # リーフノードの場合，MSEにサンプル数を乗算する
    # 左右部分木に対して再帰的に呼び出し，総MSEを合計する
    return total_mse(node['left']) + total_mse(node['right'])

def compute_alpha(node):
    # 与えられたノードに対する有効α（effective α）を計算する
    if node['is_leaf']:
        return np.inf  # リーフノードの場合，プルーニング対象外とするため無限大を返す
    R_t = node['mse'] * node['num_samples']  # ノード単体の誤差コスト
    R_T_t = total_mse(node)  # 部分木全体の誤差コスト
    T_t_leaves = count_leaves(node)  # 部分木のリーフノード数
    if T_t_leaves <= 1:
        return np.inf  # リーフノード数が1以下の場合，プルーニング対象外とする
    # 有効α = ノード単体の誤差減少量 ÷ 削除されるリーフノード数
    alpha_t = (R_t - R_T_t) / (T_t_leaves - 1)
    return alpha_t

def prune_tree(node, alpha_threshold):
    # 与えられたα閾値以下の部分木をプルーニングし，単一のリーフノードに変換する
    if node['is_leaf']:
        return  # リーフノードの場合，処理を行わない
    # 左右部分木を再帰的にプルーニング
    prune_tree(node['left'], alpha_threshold)
    prune_tree(node['right'], alpha_threshold)
    # 現ノードに対する有効αを計算
    alpha = compute_alpha(node)
    if alpha <= alpha_threshold:
        # αが閾値以下の場合，左右部分木を削除しリーフノード化する
        node['left'] = None
        node['right'] = None
        node['is_leaf'] = True

def find_min_alpha(node):
    # 木全体において最小の有効αを探索する
    if node['is_leaf']:
        return np.inf  # リーフノードの場合，プルーニング対象外とする
    alpha = compute_alpha(node)
    # 左右部分木からも最小αを再帰的に探索
    alpha_left = find_min_alpha(node['left'])
    alpha_right = find_min_alpha(node['right'])
    # 現ノード，左部分木，右部分木の最小値を返す
    return min(alpha, alpha_left, alpha_right)

### コード3.16　PythonでCARTによる回帰木構築（Step 6）

In [16]:
def predict_input(x, node):
    # 1サンプルに対して回帰木で予測値を取得する
    while not node['is_leaf']:
        # 特徴量値に基づいて左右の子ノードに移動
        if x[node['feature_index']] < node['threshold']:
            node = node['left']
        else:
            node = node['right']
    return node['predicted_value']  # リーフノードの予測値を返す

def mse_score(X, y, tree):
    # データセット全体に対する平均二乗誤差（MSE）を計算する
    predictions = [predict_input(x, tree) for x in X]
    return np.mean((y - predictions) ** 2)

### コード3.17　PythonでCARTによる回帰木構築（実行）

In [17]:
from copy import deepcopy

# データの読み込みと分割（訓練データをさらに訓練用・検証用に分割する）
X_train, X_test, y_train, y_test = load_data()
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# 回帰木の構築（訓練データのみで学習する）
tree = build_tree(X_tr, y_tr, max_depth=15, min_samples_split=5)

# プルーニング前のMSE評価
mse_train_before = mse_score(X_tr, y_tr, tree)
mse_test_before  = mse_score(X_test, y_test, tree)
print(f'Before pruning - Train MSE：{mse_train_before:.4f}, Test MSE：{mse_test_before:.4f}')

# 検証データを使ってプルーニング後の最良の木を選択する
best_tree = deepcopy(tree)
best_mse  = mse_score(X_val, y_val, tree)   # ← 検証データで評価
current_tree = deepcopy(tree)

while True:
    min_alpha = find_min_alpha(current_tree)
    if min_alpha == np.inf:
        break
    prune_tree(current_tree, min_alpha)
    mse_val_current = mse_score(X_val, y_val, current_tree)   # ← 検証データで評価
    if mse_val_current < best_mse:
        best_mse = mse_val_current
        best_tree = deepcopy(current_tree)

# プルーニング後のMSE評価（テストデータは最終評価にのみ使用する）
mse_train_after = mse_score(X_tr, y_tr, best_tree)
mse_test_after  = mse_score(X_test, y_test, best_tree)
print(f'After pruning  - Train MSE：{mse_train_after:.4f}, Test MSE：{mse_test_after:.4f}')

Before pruning - Train MSE：0.0064, Test MSE：0.0143
After pruning  - Train MSE：0.0098, Test MSE：0.0135


## 3.4.3　CHAIDによる分類木アルゴリズム

### コード3.18　PythonでCHAIDによる分類木構築（Step 0）

In [18]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency
import math

### コード3.19　PythonでCHAIDによる分類木構築（Step 1）

In [19]:
def load_data():
    # Titanic データセットを読み込み，必要な列のみを使用する
    df = sns.load_dataset('titanic').dropna(
        subset=['survived', 'sex', 'class', 'embarked', 'age', 'fare']
    )

    # 説明変数と目的変数を分離する
    X = df[['sex', 'class', 'embarked', 'age', 'fare']].copy()
    y = df['survived'].values

    # 乗船港のカテゴリ数を減らし，統計的検定が可能な形にする
    X['embarked'] = X['embarked'].replace({'Q': 'Other', 'C': 'Other'})

    # カテゴリ変数は文字列として統一的に扱う
    for col in ['sex', 'class', 'embarked']:
        X[col] = X[col].astype(str)

    # クラス比を保ったまま訓練用・テスト用に分割する
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    return (
        X_train.reset_index(drop=True),
        X_test.reset_index(drop=True),
        y_train,
        y_test
    )

### コード3.20　PythonでCHAIDによる分類木構築（Step 2）

In [20]:
def init_node(y):
    # ノードに含まれるサンプル数を記録する
    num_samples = len(y)

    # クラスごとのサンプル数を集計する
    num_positive = np.sum(y)
    num_negative = num_samples - num_positive

    # 多数決によりこのノードでの予測クラスを決定する
    predicted_class = 1 if num_positive >= num_negative else 0

    return {
        'num_samples': num_samples,
        'predicted_class': predicted_class,
        'is_leaf': False,
        'feature': None,
        'children': {},
        'split_map': None,
        'cut_point': None
    }


def merge_similar_categories(feature_series, y, alpha_merge):
    # カテゴリを段階的に統合し，統計的に区別できない水準をまとめる
    current_X = feature_series.copy()
    y_series = pd.Series(y)

    while True:
        categories = list(current_X.unique())
        if len(categories) <= 1:
            break

        best_p = -1
        pair_to_merge = None

        # すべてのカテゴリ対についてカイ二乗検定を行う
        for i in range(len(categories)):
            for j in range(i + 1, len(categories)):
                c1, c2 = categories[i], categories[j]
                mask = current_X.isin([c1, c2])
                contingency = pd.crosstab(current_X[mask], y_series[mask])

                # 2×2表にならない場合は差がないとみなす
                if contingency.shape != (2, 2):
                    p = 1.0
                else:
                    _, p, _, _ = chi2_contingency(contingency)

                if p > best_p:
                    best_p = p
                    pair_to_merge = (c1, c2)

        # 統合後も差がないと判断できる場合にカテゴリをまとめる
        if pair_to_merge and best_p > alpha_merge:
            c1, c2 = pair_to_merge
            new_name = f"{c1}+{c2}"
            current_X = current_X.replace({c1: new_name, c2: new_name})
        else:
            break

    return current_X

### コード3.21　PythonでCHAIDによる分類木構築（Step 3）

In [21]:
def find_best_cut(X_col, y):
    # 連続値特徴量に対して最も有意な分割点を探索する
    best_p = 1.1
    best_cut = None
    y_series = pd.Series(y)

    sorted_values = np.sort(X_col.unique())
    cut_points = (sorted_values[:-1] + sorted_values[1:]) / 2

    for cut in cut_points:
        # 分割点で二値化し分割表を作成する
        binned = (X_col > cut).astype(int)
        contingency = pd.crosstab(binned, y_series)

        if contingency.shape != (2, 2):
            continue

        _, p, _, _ = chi2_contingency(contingency)
        if p < best_p:
            best_p = p
            best_cut = cut

    return best_p, best_cut

def best_split(X, y, alpha_split, alpha_merge):
    # 全特徴量の中から最も有意な分割を選択する
    best_p = 1.1
    best_feature = None
    result_info = None

    y_series = pd.Series(y)

    for col in X.columns:
        # 数値特徴量とカテゴリ特徴量を分けて処理する
        if np.issubdtype(X[col].dtype, np.number):
            p, cut = find_best_cut(X[col], y)
            adj_p = p
            info = cut
        else:
            merged_series = merge_similar_categories(X[col], y, alpha_merge)
            if merged_series.nunique() < 2:
                continue

            contingency = pd.crosstab(merged_series, y_series)
            if (contingency.sum(axis=1) < 5).any():
                continue

            _, p, _, _ = chi2_contingency(contingency)
            adj_p = min(p * math.comb(X[col].nunique(), 2), 1.0)
            info = merged_series

        if adj_p < best_p:
            best_p = adj_p
            best_feature = col
            result_info = info

    # 有意な分割が見つからない場合は分割しない
    if best_p >= alpha_split:
        return None, None

    return best_feature, result_info

### コード3.22　PythonでCHAIDによる分類木構築（Step 4）

In [22]:
def build_tree(
    X, y, depth=0, max_depth=3,
    alpha_split=0.05, alpha_merge=0.05,
    min_samples_split=20
):
    node = init_node(y)

    # 深さ，純度，サンプル数による停止条件
    if depth >= max_depth or len(np.unique(y)) == 1 or len(y) < min_samples_split:
        node['is_leaf'] = True
        return node

    feature, split_info = best_split(X, y, alpha_split, alpha_merge)
    if feature is None:
        node['is_leaf'] = True
        return node

    node['feature'] = feature

    # 連続値特徴量の場合は二分割する
    if isinstance(split_info, (float, int, np.floating)):
        node['cut_point'] = split_info

        left = X[feature] <= split_info
        right = X[feature] > split_info

        node['children']['<='] = build_tree(
            X[left].reset_index(drop=True), y[left],
            depth + 1, max_depth,
            alpha_split, alpha_merge, min_samples_split
        )
        node['children']['>'] = build_tree(
            X[right].reset_index(drop=True), y[right],
            depth + 1, max_depth,
            alpha_split, alpha_merge, min_samples_split
        )
    else:
        # カテゴリ特徴量の場合はグループごとに分岐する
        node['split_map'] = dict(zip(X[feature], split_info))
        for g in split_info.unique():
            mask = (split_info == g).values
            node['children'][g] = build_tree(
                X[mask].reset_index(drop=True), y[mask],
                depth + 1, max_depth,
                alpha_split, alpha_merge, min_samples_split
            )

    return node

### コード3.23　PythonでCHAIDによる分類木構築（Step 5）

In [23]:
def predict_input(x, node):
    # ルートから葉まで分岐条件に従って辿る
    while not node['is_leaf']:
        feat = node['feature']
        val = x[feat]

        if node['cut_point'] is not None:
            node = node['children']['<=' if val <= node['cut_point'] else '>']
        else:
            key = node['split_map'].get(val)
            if key not in node['children']:
                break
            node = node['children'][key]

    return node['predicted_class']


def calculate_accuracy(X, y, tree):
    # 各サンプルに対する予測結果から正解率を計算する
    preds = [predict_input(X.iloc[i], tree) for i in range(len(X))]
    return np.mean(preds == y)

### コード3.24　PythonでCHAIDによる分類木構築（実行）

In [24]:
X_train, X_test, y_train, y_test = load_data()

tree = build_tree(
    X_train, y_train,
    max_depth=3,
    alpha_split=0.05,
    alpha_merge=0.05,
    min_samples_split=20
)

print(f"Train Accuracy: {calculate_accuracy(X_train, y_train, tree):.4f}")
print(f"Test Accuracy : {calculate_accuracy(X_test, y_test, tree):.4f}")

Train Accuracy: 0.8207
Test Accuracy : 0.8112


## 3.4.3　ID3による分類木アルゴリズム

### コード3.25　PythonでID3による分類木構築（Step 0）

In [25]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from collections import Counter

### コード3.26　PythonでID3による分類木構築（Step 1）

In [26]:
def load_data():
    # Titanic データセットを読み込む
    df = sns.load_dataset('titanic').dropna(subset=['embarked', 'class', 'sex', 'age', 'fare'])

    # 特徴量として sex, class, embarked を抽出
    X = df[['sex', 'class', 'embarked']] .copy()
    y = df['survived'].values

    # 特徴量をカテゴリ型に変換
    for col in X.columns:
        X[col] = X[col].astype('category')

    # データを層化サンプリングでトレーニング用・テスト用に分割
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    return X_train, X_test, y_train, y_test

### コード3.27　PythonでID3による分類木構築（Step 2）

In [27]:
# エントロピーを計算する関数
# y：ターゲット変数の値（0,1ラベルの配列）
def entropy(y):
    counts = np.bincount(y)  # クラスごとの出現回数を数える
    probs = counts / len(y)  # 各クラスの出現割合を計算
    # 各クラスの p * log2(p) を合計（p>0のときのみ）し，負の和を返す
    return -np.sum([p * np.log2(p) for p in probs if p > 0])

# 情報利得を計算する関数
# y：元のターゲット変数，groups：特徴量のカテゴリ値ごとの y 配列リスト
def information_gain(y, groups):
    H_before = entropy(y)  # 分割前のエントロピー
    H_after = sum(len(group) / len(y) * entropy(group) for group in groups)  # 分割後の重み付きエントロピー
    return H_before - H_after  # 情報利得 = 分割前 - 分割後

### コード3.28　PythonでID3による分類木構築（Step 3）

In [28]:
# 最適な分割特徴量を決定する関数
# X：特徴量データフレーム，y：ターゲット変数
def best_split(X, y):
    best_score, best_feature = -1, None  # 情報利得の最大値と対応する特徴量名を初期化
    for col in X.columns:
        # 特徴量 col の各カテゴリ値ごとに y を分けてリスト化
        groups = [y[X[col] == val] for val in X[col].unique()]
        if len(groups) < 2:
            continue  # グループが1つしかない場合はスキップ
        ig = information_gain(y, groups)  # 情報利得を計算
        if ig > best_score:
            best_score, best_feature = ig, col  # 最大情報利得を更新
    return best_feature  # 最も情報利得が大きい特徴量名を返す

### コード3.29　PythonでID3による分類木構築（Step 4）

In [29]:
def build_tree(X, y, depth=0, max_depth=5):
    counts = Counter(y)  # クラスごとの出現回数を数える
    majority = counts.most_common(1)[0][0]  # 最頻クラス（多数決）を取得

    # ノード情報を辞書に保存
    node = {'depth': depth, 'majority': majority}

    # --- 終了条件 ---
    # 最大深さに達した，または全サンプルが同じクラスの場合，リーフノードとする
    if depth >= max_depth or len(set(y)) == 1:
        node['is_leaf'] = True
        return node

    # 最適な分割特徴量を取得（Step 3 の関数）
    feature = best_split(X, y)
# すべての特徴量について有効な分割が存在しない場合，リーフノードとする
    if feature is None:
        node['is_leaf'] = True
        return node

    # --- 子ノードの生成 ---
    node['is_leaf'] = False
    node['feature'] = feature  # 分割に使う特徴量を記録
    node['children'] = {}  # 子ノードを格納する辞書を用意

    # 特徴量の各カテゴリ値ごとにデータを分け，再帰的に子ノードを作成
    for val in X[feature].unique():
        mask = X[feature] == val  # 現在のカテゴリ値の行を抽出
        child = build_tree(X[mask], y[mask], depth + 1, max_depth)  # 再帰呼び出し
        node['children'][val] = child  # 子ノードを辞書に登録

    return node  # 作成したノードを返す

### コード3.30　PythonでID3による分類木構築（Step 5）

In [30]:
def predict_one(x, node):
    # 1つのサンプル（x）に対して木をたどって予測を行う関数
    while not node['is_leaf']:  # リーフノードに到達するまでループする
        val = x[node['feature']]  # 現在のノードで使う特徴量の値を取得する
        if val in node['children']:  # その値に対応する子ノードが存在する場合
            node = node['children'][val]  # 子ノードに移動する
        else:
            break  # 未知の値（訓練データに存在しなかったカテゴリ値）の場合，ループを抜ける
    return node['majority']  # 最終的に到達したノードの多数派クラスを返す

def evaluate(X, y, tree):
    # データセット X, y に対して木（tree）を使って精度を評価する関数
    preds = [predict_one(X.iloc[i], tree) for i in range(len(X))]
    # 各行（サンプル）について predict_one で予測を行い，リストに保存する
    return np.mean(preds == y)
    # 予測結果と正解 y を比較し，True（正解）の平均値＝正解率を計算して返す

### コード3.31　PythonでID3による分類木構築（実行）

In [31]:
X_train, X_test, y_train, y_test = load_data()
print("\nMethod: ID3")
tree = build_tree(X_train, y_train, max_depth=5)
acc_train = evaluate(X_train, y_train, tree)
acc_test = evaluate(X_test, y_test, tree)
print(f"Train Accuracy: {acc_train:.4f}")
print(f"Test Accuracy: {acc_test:.4f}")


Method: ID3
Train Accuracy: 0.8049
Test Accuracy: 0.7762
